# LIDC-IDRI Dataset Profiler — standalone

**Needs nothing but the LIDC dataset.** No internet, no repo clone, no code
dataset, no GPU. Everything is in cell 2.

## Setup
1. **Add Data** → search `lidcidri` → add **zhangweiled/lidcidri**
2. **Settings → Accelerator → None** (CPU only; don't spend GPU quota)
3. Internet can stay **off**
4. Run all cells

## What it measures
1. positive pixel fraction, per reader-vote threshold
2. empty-merged-mask fraction, per vote threshold
3. reader-mask histogram — slices with fewer masks than the vote needs can
   never be positive, and silently inflate the negative class
4. native resolution, before any resize
5. nodule size distribution native **and** after the 128×128 training resize,
   with a survival table
6. slice / patient / nodule counts, all nodules vs `nodule-0` only
7. **inter-reader pairwise Dice** — roughly the ceiling on achievable Dice,
   since no model can meaningfully beat the agreement between the
   radiologists who drew the labels

Item 7 is the one to read first. If it comes back near 0.85 while the earlier
notebook reported **0.8826** on a slice-level split, that model was scoring at
or above the experts' agreement with each other — i.e. fitting the vote-≥2
fusion rule rather than the anatomy.

In [ ]:
# ── 1. Locate the dataset ────────────────────────────────────────────────
import pathlib

INPUT = pathlib.Path("/kaggle/input")
print("contents of /kaggle/input:")
for d in sorted(INPUT.iterdir()):
    kids = sorted(p.name for p in d.iterdir())[:8] if d.is_dir() else []
    print("  ", d.name, "->", kids if kids else "(file)")

In [ ]:
# ======================================================================
# LIDC-IDRI dataset profiler -- STANDALONE (generated; do not edit)
#
# Regenerate with:  python build/make_standalone.py
#
# Needs only numpy, OpenCV and matplotlib, all preinstalled on Kaggle.
# No internet, no repo clone, no extra dataset -- just the LIDC slices.
#
# Measures:
#   1. positive pixel fraction, per reader-vote threshold
#   2. empty-merged-mask fraction, per vote threshold
#   3. reader-mask histogram
#   4. native resolution (before any resize)
#   5. nodule size distribution, native AND after the training resize
#   6. slice/patient/nodule counts, all nodules vs nodule-0 only
#   7. inter-reader pairwise Dice -- the achievable-Dice ceiling
# ======================================================================

# Must stay first: the stitched bodies annotate with names defined later.
from __future__ import annotations

import json
import os
import re
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, asdict
from pathlib import Path
from types import SimpleNamespace

import cv2
import numpy as np






@dataclass(frozen=True)
class SliceRecord:
    patient: str      # "LIDC-IDRI-0001" -- the grouping key for splits
    nodule: str       # "nodule-0"
    slice_name: str   # "0.png"
    image: str        # absolute path
    masks: tuple[str, ...]  # absolute paths, one per reader that has this slice

    @property
    def uid(self) -> str:
        return f"{self.patient}/{self.nodule}/{self.slice_name}"


def _natural_key(s: str):
    """Sort "10.png" after "9.png" so slice order is anatomical."""
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]


# Shared with zip_source so both indexers accept exactly the same files.
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

# A patient directory is "LIDC-IDRI-" plus digits and nothing else. Matching
# the prefix alone also matches the dataset's own "LIDC-IDRI-slices" wrapper,
# which then gets treated as a patient with no nodules.
_PATIENT_RE = re.compile(r"^LIDC-IDRI-\d+$")


def is_patient_dir(p: Path) -> bool:
    return p.is_dir() and bool(_PATIENT_RE.match(p.name))


def find_dataset_root(root: str | Path) -> Path:
    """Resolve the directory that directly contains LIDC-IDRI-* folders.

    Kaggle mount paths vary. `kagglehub.dataset_download` mounts under
    `/kaggle/input/datasets/<owner>/<slug>/`, which is what the original
    notebook hard-coded and it was correct for that setup. Adding the same
    dataset through the Add Data panel instead gives `/kaggle/input/<slug>/`,
    and either may or may not nest a `LIDC-IDRI-slices/` directory.

    So rather than hard-code any of them, search a few levels down for
    directories matching `LIDC-IDRI-<number>`. This is about portability
    across mount styles, not about the old path being broken.
    """
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(
            f"dataset root does not exist: {root}\n"
            "On Kaggle, add the 'zhangweiled/lidcidri' dataset to the notebook and "
            "pass --set data.root=/kaggle/input/<mounted-name>"
        )

    def has_patients(d: Path) -> bool:
        try:
            return any(is_patient_dir(p) for p in d.iterdir())
        except (PermissionError, OSError):
            return False

    if has_patients(root):
        return root
    for depth in range(1, 4):
        for cand in sorted(root.glob("/".join(["*"] * depth))):
            if cand.is_dir() and has_patients(cand):
                return cand
    raise FileNotFoundError(
        f"no LIDC-IDRI-<number> patient directories found under {root} "
        "(searched 3 levels deep)"
    )


def build_index(
    root: str | Path,
    nodules: list[str] | None = None,
    n_readers: int = 4,
) -> list[SliceRecord]:
    """Scan the tree and return one record per slice.

    `nodules=None` takes every nodule directory per patient. The notebook only
    read `nodule-0`, which silently dropped the second and later nodules of
    every multi-nodule patient.
    """
    root = find_dataset_root(root)
    records: list[SliceRecord] = []

    patients = sorted(
        (p for p in root.iterdir() if is_patient_dir(p)),
        key=lambda p: _natural_key(p.name),
    )

    for pdir in patients:
        if nodules is None:
            ndirs = sorted(
                (d for d in pdir.iterdir() if d.is_dir() and d.name.startswith("nodule-")),
                key=lambda d: _natural_key(d.name),
            )
        else:
            ndirs = [pdir / n for n in nodules]

        for ndir in ndirs:
            img_dir = ndir / "images"
            if not img_dir.is_dir():
                continue

            mask_dirs = [ndir / f"mask-{i}" for i in range(n_readers)]
            mask_dirs = [d for d in mask_dirs if d.is_dir()]
            if not mask_dirs:
                continue

            for img in sorted(img_dir.iterdir(), key=lambda p: _natural_key(p.name)):
                if img.suffix.lower() not in IMAGE_SUFFIXES:
                    continue
                masks = tuple(str(d / img.name) for d in mask_dirs if (d / img.name).exists())
                if not masks:
                    continue
                records.append(
                    SliceRecord(
                        patient=pdir.name,
                        nodule=ndir.name,
                        slice_name=img.name,
                        image=str(img),
                        masks=masks,
                    )
                )

    if not records:
        raise RuntimeError(f"index is empty -- found no usable slices under {root}")
    return records


def index_stats(records: list[SliceRecord]) -> dict:
    patients = {r.patient for r in records}
    nodules = {(r.patient, r.nodule) for r in records}
    per_reader_counts: dict[int, int] = {}
    for r in records:
        per_reader_counts[len(r.masks)] = per_reader_counts.get(len(r.masks), 0) + 1
    return {
        "slices": len(records),
        "patients": len(patients),
        "nodules": len(nodules),
        "slices_per_patient": round(len(records) / max(len(patients), 1), 2),
        "mask_count_histogram": dict(sorted(per_reader_counts.items())),
    }


def save_index(records: list[SliceRecord], path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {"stats": index_stats(records), "records": [asdict(r) for r in records]}
    path.write_text(json.dumps(payload, indent=1), encoding="utf-8")


def load_index(path: str | Path) -> list[SliceRecord]:
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    return [
        SliceRecord(
            patient=d["patient"],
            nodule=d["nodule"],
            slice_name=d["slice_name"],
            image=d["image"],
            masks=tuple(d["masks"]),
        )
        for d in payload["records"]
    ]






# Equivalent-diameter bins in pixels, for the size-stratified metrics that
# RESEARCH_NOTES section 6 item 4 calls for. Kept here so the profiler and the
# later evaluation use identical bins.
SIZE_BINS_PX = [0, 4, 8, 16, 32, 64, np.inf]
SIZE_BIN_LABELS = ["<4", "4-8", "8-16", "16-32", "32-64", ">=64"]

# "Is this nodule still there after the resize?" thresholds, in pixels of area.
SURVIVAL_THRESHOLDS = [1, 4, 9, 16, 25]


@dataclass
class SliceProfile:
    """Per-slice measurements. Scalars only -- no masks are retained."""

    patient: str
    nodule: str
    height: int
    width: int
    n_masks: int
    # Positive pixels for each reader present, at native resolution.
    reader_px: tuple[int, ...]
    # Positive pixels of the merged mask at each vote threshold 1..n_readers.
    vote_px: dict[int, int]
    total_px: int
    # Connected components of the merged mask at the configured threshold.
    component_areas: tuple[int, ...]
    # Same mask binarised then resized to img_size, as the pipeline does.
    resized_px: int
    resized_component_areas: tuple[int, ...]
    # Mean pairwise Dice between readers; None when fewer than 2 masks exist.
    mean_pairwise_dice: float | None
    # Each reader vs the others' majority consensus; None with fewer than 3.
    reader_vs_consensus_dice: float | None


def _dice(a: np.ndarray, b: np.ndarray) -> float:
    """Dice between two boolean masks; both-empty counts as perfect."""
    a, b = a.astype(bool), b.astype(bool)
    denom = a.sum() + b.sum()
    return 1.0 if denom == 0 else 2.0 * np.count_nonzero(a & b) / denom


def _reader_vs_consensus_dice(masks: list[np.ndarray], vote: int) -> float | None:
    """Leave-one-reader-out: each reader scored against the others' consensus.

    This is the *correct* ceiling for a model trained on a fused target, and it
    is higher than mean pairwise agreement. Predicting a consensus is easier
    than predicting an individual, because fusing several readers averages out
    each one's idiosyncratic noise -- so comparing a model's score against the
    fused mask to reader-vs-reader Dice understates what is achievable.

    Here reader r is compared against the majority vote of the remaining
    readers, which is the same kind of quantity the model is asked to produce.
    """
    if len(masks) < 3:
        return None
    scores = []
    for r in range(len(masks)):
        others = [m for i, m in enumerate(masks) if i != r]
        # Scale the vote threshold to the smaller panel, keeping "majority".
        thr = max(1, min(vote, len(others)))
        consensus = (np.sum(others, axis=0) >= thr)
        scores.append(_dice(masks[r], consensus))
    return float(np.mean(scores))


def _pairwise_dice(masks: list[np.ndarray]) -> float | None:
    """Mean Dice over every pair of reader masks -- raw annotator agreement."""
    if len(masks) < 2:
        return None
    scores = []
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            # Two readers who both marked nothing agree perfectly.
            scores.append(_dice(masks[i], masks[j]))
    return float(np.mean(scores))


def _components(mask: np.ndarray) -> tuple[int, ...]:
    """Areas of the 8-connected foreground components, largest first."""
    if not mask.any():
        return ()
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    areas = sorted((int(stats[k, cv2.CC_STAT_AREA]) for k in range(1, n)), reverse=True)
    return tuple(areas)


def profile_slice(
    rec: SliceRecord,
    img_size: int,
    vote_threshold: int,
    n_readers: int,
    source=None,
) -> SliceProfile:
    """Measure one slice.

    `source` is an optional ZipSource; when given, members are decoded from
    the zip instead of read from the filesystem. Both paths produce identical
    arrays, so every number below is unaffected by the choice.
    """
    read = source.imread_gray if source is not None else (
        lambda p: cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    )

    img = read(rec.image)
    if img is None:
        raise FileNotFoundError(f"could not read image: {rec.image}")
    h, w = img.shape[:2]

    masks: list[np.ndarray] = []
    for mp in rec.masks:
        m = read(mp)
        if m is None:
            continue
        if m.shape[:2] != (h, w):
            # A reader mask that disagrees with its image on shape would
            # silently misalign; resize it to the image and keep going.
            m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
        masks.append((m > 127).astype(np.uint8))

    votes = np.zeros((h, w), dtype=np.uint8)
    for m in masks:
        votes += m

    vote_px = {t: int(np.count_nonzero(votes >= t)) for t in range(1, n_readers + 1)}

    merged = (votes >= vote_threshold).astype(np.uint8)

    # Binarise then nearest-resize -- exactly what data.preprocess does, so the
    # survival numbers describe the masks the model actually trains on.
    resized = (
        cv2.resize(merged, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        if (h, w) != (img_size, img_size)
        else merged
    )

    return SliceProfile(
        patient=rec.patient,
        nodule=rec.nodule,
        height=h,
        width=w,
        n_masks=len(masks),
        reader_px=tuple(int(m.sum()) for m in masks),
        vote_px=vote_px,
        total_px=h * w,
        component_areas=_components(merged),
        resized_px=int(np.count_nonzero(resized)),
        resized_component_areas=_components(resized),
        mean_pairwise_dice=_pairwise_dice(masks),
        reader_vs_consensus_dice=_reader_vs_consensus_dice(masks, vote_threshold),
    )


# ------------------------------------------------------------------ aggregate


def _pct(values, qs=(5, 25, 50, 75, 95)) -> dict:
    if len(values) == 0:
        return {}
    a = np.asarray(values, dtype=np.float64)
    return {f"p{q}": round(float(np.percentile(a, q)), 3) for q in qs}


def _describe(values, qs=(5, 25, 50, 75, 95)) -> dict:
    if len(values) == 0:
        return {"n": 0}
    a = np.asarray(values, dtype=np.float64)
    return {
        "n": int(a.size),
        "mean": round(float(a.mean()), 4),
        "std": round(float(a.std()), 4),
        "min": round(float(a.min()), 4),
        "max": round(float(a.max()), 4),
        **_pct(a, qs),
    }


def aggregate(profiles: list[SliceProfile], cfg, n_readers: int) -> dict:
    size = cfg.data.img_size
    vt = cfg.data.vote_threshold
    n = len(profiles)

    # (4) native resolution
    shapes = Counter((p.height, p.width) for p in profiles)
    resolution = {
        "distinct_shapes": len(shapes),
        "shapes": [{"h": h, "w": w, "slices": c} for (h, w), c in shapes.most_common(10)],
        "uniform": len(shapes) == 1,
    }

    # (3) reader-mask histogram
    mask_hist = Counter(p.n_masks for p in profiles)
    reader_masks = {
        "histogram": {str(k): mask_hist[k] for k in sorted(mask_hist)},
        "fraction_with_all_readers": round(mask_hist[n_readers] / n, 4) if n else 0.0,
        # Below the vote threshold a slice can never be positive: its label is
        # all-background by construction, not by anatomy.
        "slices_below_vote_threshold": sum(c for k, c in mask_hist.items() if k < vt),
        "fraction_below_vote_threshold": round(
            sum(c for k, c in mask_hist.items() if k < vt) / n, 4
        )
        if n
        else 0.0,
    }

    # (1) + (2) vote-threshold sensitivity
    total_px = sum(p.total_px for p in profiles)
    vote_rows = []
    for t in range(1, n_readers + 1):
        pos = sum(p.vote_px[t] for p in profiles)
        empty = sum(1 for p in profiles if p.vote_px[t] == 0)
        vote_rows.append(
            {
                "vote_threshold": t,
                "positive_pixel_fraction": round(pos / total_px, 8) if total_px else 0.0,
                "positive_pixels": pos,
                "empty_slices": empty,
                "empty_slice_fraction": round(empty / n, 4) if n else 0.0,
                "imbalance_ratio": round((total_px - pos) / pos, 1) if pos else None,
            }
        )

    # (5) nodule size distribution, native and resized
    native_areas = [a for p in profiles for a in p.component_areas]
    resized_areas = [a for p in profiles for a in p.resized_component_areas]
    # Equivalent diameter of a circle with the same area.
    native_diam = [2.0 * np.sqrt(a / np.pi) for a in native_areas]
    resized_diam = [2.0 * np.sqrt(a / np.pi) for a in resized_areas]

    diam_hist, _ = np.histogram(native_diam, bins=SIZE_BINS_PX) if native_diam else (np.zeros(6, int), None)
    rdiam_hist, _ = np.histogram(resized_diam, bins=SIZE_BINS_PX) if resized_diam else (np.zeros(6, int), None)

    survival = []
    for thr in SURVIVAL_THRESHOLDS:
        before = sum(1 for a in native_areas if a >= thr)
        after = sum(1 for a in resized_areas if a >= thr)
        survival.append(
            {
                "min_area_px": thr,
                "components_native": before,
                "components_resized": after,
                "retained_fraction": round(after / before, 4) if before else None,
            }
        )

    lost = sum(1 for p in profiles if p.component_areas and not p.resized_component_areas)

    sizes = {
        "native_component_area_px": _describe(native_areas),
        "resized_component_area_px": _describe(resized_areas),
        "native_equiv_diameter_px": _describe(native_diam),
        "resized_equiv_diameter_px": _describe(resized_diam),
        "diameter_histogram_native": dict(zip(SIZE_BIN_LABELS, [int(x) for x in diam_hist])),
        "diameter_histogram_resized": dict(zip(SIZE_BIN_LABELS, [int(x) for x in rdiam_hist])),
        "components_total_native": len(native_areas),
        "components_total_resized": len(resized_areas),
        "resize_survival": survival,
        "slices_losing_all_foreground_on_resize": lost,
        "fraction_losing_all_foreground": round(lost / n, 5) if n else 0.0,
        "components_per_nonempty_slice": round(
            len(native_areas) / max(sum(1 for p in profiles if p.component_areas), 1), 3
        ),
    }

    # Inter-reader agreement -- the practical Dice ceiling.
    agreements = [p.mean_pairwise_dice for p in profiles if p.mean_pairwise_dice is not None]
    agreement = {
        **_describe(agreements),
        "note": "mean pairwise Dice between radiologists -- raw annotator "
        "agreement, NOT the ceiling for a model predicting a fused target",
    }

    loo = [p.reader_vs_consensus_dice for p in profiles
           if p.reader_vs_consensus_dice is not None]
    consensus_ceiling = {
        **_describe(loo),
        "note": "each reader vs the majority consensus of the others -- the "
        "correct ceiling for a model trained on the fused mask, since "
        "predicting a consensus is easier than predicting an individual",
    }

    return {
        "resolution": resolution,
        "reader_masks": reader_masks,
        "vote_sensitivity": vote_rows,
        "configured_vote_threshold": vt,
        "train_img_size": size,
        "sizes": sizes,
        "inter_reader_agreement": agreement,
        "consensus_ceiling": consensus_ceiling,
    }


# -------------------------------------------------------------------- report


def _md_table(headers: list[str], rows: list[list]) -> str:
    out = ["| " + " | ".join(headers) + " |", "|" + "|".join(["---"] * len(headers)) + "|"]
    for r in rows:
        out.append("| " + " | ".join("" if v is None else str(v) for v in r) + " |")
    return "\n".join(out)


def write_report(result: dict, out_dir: Path) -> Path:
    counts = result["counts"]
    agg = result["measurements"]
    size = agg["train_img_size"]
    vt = agg["configured_vote_threshold"]

    L: list[str] = [
        "# Dataset profile",
        "",
        f"Root: `{result['root']}`  ",
        f"Generated: {result['generated']}  ",
        f"Slices profiled: **{result['slices_profiled']}**"
        + (f" (sampled from {counts['all_nodules']['slices']})" if result["sampled"] else "")
        + "  ",
        f"Training resize target: {size}x{size}, vote threshold >={vt}",
        "",
        "## 6. Counts: all nodules vs nodule-0 only",
        "",
    ]

    a, z = counts["all_nodules"], counts["nodule_0_only"]
    L.append(
        _md_table(
            ["scope", "slices", "patients", "nodules", "slices/patient"],
            [
                ["all nodules", a["slices"], a["patients"], a["nodules"], a["slices_per_patient"]],
                ["`nodule-0` only", z["slices"], z["patients"], z["nodules"], z["slices_per_patient"]],
            ],
        )
    )
    extra = a["slices"] - z["slices"]
    pct = 100.0 * extra / z["slices"] if z["slices"] else 0.0
    L += [
        "",
        f"Using every nodule adds **{extra} slices ({pct:+.1f}%)** and "
        f"{a['nodules'] - z['nodules']} nodules over the notebook's `nodule-0` filter.",
        "",
        "## 4. Native resolution",
        "",
    ]

    res = agg["resolution"]
    L.append(
        _md_table(
            ["height", "width", "slices"],
            [[s["h"], s["w"], s["slices"]] for s in res["shapes"]],
        )
    )
    L += ["", f"Distinct shapes: {res['distinct_shapes']} "
          f"({'uniform' if res['uniform'] else 'MIXED -- resize behaviour varies per slice'})", ""]

    if not res["uniform"]:
        L += [
            "> Mixed native resolutions mean a fixed resize changes scale "
            "non-uniformly across slices, so pixel areas are not comparable "
            "between patients without normalising by native shape.",
            "",
        ]

    L += ["## 3. Reader-mask histogram", ""]
    rm = agg["reader_masks"]
    L.append(_md_table(["masks present", "slices"], [[k, v] for k, v in rm["histogram"].items()]))
    L += [
        "",
        f"Slices with the full reader set: **{100 * rm['fraction_with_all_readers']:.1f}%**  ",
        f"Slices with fewer than {vt} masks: **{rm['slices_below_vote_threshold']} "
        f"({100 * rm['fraction_below_vote_threshold']:.2f}%)**",
        "",
    ]
    if rm["slices_below_vote_threshold"]:
        L += [
            f"> These cannot be positive under a >={vt} vote regardless of anatomy -- "
            "their label is all-background by construction. They inflate the "
            "negative class and should probably be dropped rather than trained on.",
            "",
        ]

    L += ["## 1 + 2. Vote threshold: imbalance and empty masks", ""]
    L.append(
        _md_table(
            ["vote >=", "positive pixel frac", "bg:fg ratio", "empty slices", "empty frac"],
            [
                [
                    r["vote_threshold"],
                    f"{r['positive_pixel_fraction']:.6f}",
                    r["imbalance_ratio"],
                    r["empty_slices"],
                    f"{r['empty_slice_fraction']:.4f}",
                ]
                for r in agg["vote_sensitivity"]
            ],
        )
    )
    cur = next(r for r in agg["vote_sensitivity"] if r["vote_threshold"] == vt)
    L += [
        "",
        f"At the configured threshold (>={vt}): **{100 * cur['positive_pixel_fraction']:.4f}%** of "
        f"pixels are nodule, a background:foreground ratio of about "
        f"**{cur['imbalance_ratio']}:1**, and "
        f"**{100 * cur['empty_slice_fraction']:.1f}%** of slices have an empty mask.",
        "",
        "> This is why pixel accuracy is meaningless here: predicting all "
        "background scores "
        f"{100 * (1 - cur['positive_pixel_fraction']):.3f}%.",
        "",
        "## 5. Nodule size distribution",
        "",
    ]

    s = agg["sizes"]
    L.append(
        _md_table(
            ["metric", "n", "mean", "p5", "p25", "p50", "p75", "p95", "max"],
            [
                [
                    name,
                    d.get("n"),
                    d.get("mean"),
                    d.get("p5"),
                    d.get("p25"),
                    d.get("p50"),
                    d.get("p75"),
                    d.get("p95"),
                    d.get("max"),
                ]
                for name, d in (
                    ("area px (native)", s["native_component_area_px"]),
                    (f"area px ({size}x{size})", s["resized_component_area_px"]),
                    ("equiv diameter px (native)", s["native_equiv_diameter_px"]),
                    (f"equiv diameter px ({size}x{size})", s["resized_equiv_diameter_px"]),
                )
            ],
        )
    )

    L += ["", "### Equivalent-diameter histogram (px)", ""]
    L.append(
        _md_table(
            ["diameter bin", "native", f"after {size}x{size} resize"],
            [
                [lab, s["diameter_histogram_native"].get(lab, 0), s["diameter_histogram_resized"].get(lab, 0)]
                for lab in SIZE_BIN_LABELS
            ],
        )
    )

    L += ["", "### Does the resize destroy small nodules?", ""]
    L.append(
        _md_table(
            ["min area px", "components native", "after resize", "retained"],
            [
                [
                    r["min_area_px"],
                    r["components_native"],
                    r["components_resized"],
                    f"{100 * r['retained_fraction']:.1f}%" if r["retained_fraction"] is not None else None,
                ]
                for r in s["resize_survival"]
            ],
        )
    )
    L += [
        "",
        f"Slices that had foreground natively and none after resize: "
        f"**{s['slices_losing_all_foreground_on_resize']} "
        f"({100 * s['fraction_losing_all_foreground']:.3f}%)**  ",
        f"Components per non-empty slice: {s['components_per_nonempty_slice']}",
        "",
        "## Inter-reader agreement",
        "",
    ]

    ag = agg["inter_reader_agreement"]
    cc = agg.get("consensus_ceiling", {})
    if ag.get("n"):
        L += [
            _md_table(
                ["quantity", "mean", "sd", "p5", "median", "p95"],
                [
                    ["reader vs reader (pairwise)", f"{ag['mean']:.4f}", f"{ag['std']:.4f}",
                     ag.get("p5"), ag.get("p50"), ag.get("p95")],
                ] + ([
                    ["reader vs others' consensus", f"{cc['mean']:.4f}", f"{cc['std']:.4f}",
                     cc.get("p5"), cc.get("p50"), cc.get("p95")],
                ] if cc.get("n") else []),
            ),
            "",
            "**Read the second row, not the first.** Pairwise reader agreement "
            "measures how much the annotators disagree with each other. It is "
            "not the ceiling for a model, because the model is trained against "
            f"the vote->={vt} *fusion* of all readers, and predicting a consensus "
            "is easier than predicting any individual -- fusing averages out "
            "each reader's idiosyncratic noise.",
            "",
            "The second row is the right comparison: each reader scored against "
            "the majority consensus of the others, which is the same kind of "
            "target the model produces.",
        ]
        if cc.get("n"):
            L += [
                "",
                f"So a model scoring much above **{cc['mean']:.3f}** per-image Dice "
                "on this target is doing something a radiologist cannot do against "
                "their colleagues' consensus -- which points at leakage, at an "
                "easier metric (global rather than per-image Dice), or at fitting "
                "the fusion rule rather than the anatomy.",
            ]
    else:
        L.append("Not computable -- fewer than 2 reader masks available.")

    L += ["", "---", "", "Generated by `python -m lidcseg.profile_data`.", ""]

    path = out_dir / "profile.md"
    path.write_text("\n".join(L), encoding="utf-8")
    return path


def write_figures(profiles: list[SliceProfile], agg: dict, out_dir: Path) -> None:
    try:
        import matplotlib

        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("[profile] matplotlib unavailable -- skipping figures")
        return

    size = agg["train_img_size"]
    native = [a for p in profiles for a in p.component_areas]
    resized = [a for p in profiles for a in p.resized_component_areas]

    fig, axes = plt.subplots(2, 2, figsize=(13, 8))

    ax = axes[0, 0]
    if native:
        nd = [2 * np.sqrt(a / np.pi) for a in native]
        rd = [2 * np.sqrt(a / np.pi) for a in resized] or [0]
        bins = np.linspace(0, max(max(nd), max(rd)) or 1, 50)
        ax.hist(nd, bins=bins, alpha=0.65, label="native")
        ax.hist(rd, bins=bins, alpha=0.65, label=f"{size}x{size}")
        ax.set_xlabel("equivalent diameter (px)")
        ax.set_ylabel("components")
        ax.set_title("Nodule size distribution")
        ax.legend()
    ax.grid(alpha=0.3)

    ax = axes[0, 1]
    rows = agg["vote_sensitivity"]
    ts = [r["vote_threshold"] for r in rows]
    ax.plot(ts, [r["positive_pixel_fraction"] for r in rows], marker="o", label="positive px frac")
    ax.set_yscale("log")
    ax.set_xlabel("reader vote threshold")
    ax.set_ylabel("positive pixel fraction (log)")
    ax.set_xticks(ts)
    ax2 = ax.twinx()
    ax2.plot(ts, [r["empty_slice_fraction"] for r in rows], marker="s", color="#C44E52",
             label="empty slice frac")
    ax2.set_ylabel("empty slice fraction")
    ax.set_title("Vote threshold sensitivity")
    ax.grid(alpha=0.3)
    lines = ax.get_lines() + ax2.get_lines()
    ax.legend(lines, [l.get_label() for l in lines], fontsize=8)

    ax = axes[1, 0]
    hist = agg["reader_masks"]["histogram"]
    ax.bar(list(hist.keys()), list(hist.values()), color="#55A868")
    ax.set_xlabel("reader masks present")
    ax.set_ylabel("slices")
    ax.set_title("Reader-mask availability")
    ax.grid(axis="y", alpha=0.3)

    ax = axes[1, 1]
    ags = [p.mean_pairwise_dice for p in profiles if p.mean_pairwise_dice is not None]
    if ags:
        ax.hist(ags, bins=40, color="#8172B2")
        ax.axvline(float(np.mean(ags)), color="#C44E52", ls="--",
                   label=f"mean {np.mean(ags):.3f}")
        ax.legend()
    ax.set_xlabel("mean pairwise Dice between readers")
    ax.set_ylabel("slices")
    ax.set_title("Inter-reader agreement (Dice ceiling)")
    ax.grid(alpha=0.3)

    fig.tight_layout()
    fig.savefig(out_dir / "profile.png", dpi=140)
    plt.close(fig)
    print(f"[profile] figures -> {out_dir / 'profile.png'}")


# ---------------------------------------------------------------------- main


def default_workers() -> int:
    """Thread count for the mask reads.

    I/O-bound against a network filesystem, so useful parallelism far exceeds
    the core count; capped at 32 so a huge machine does not thrash the mount.
    """
    return max(4, min(32, (os.cpu_count() or 4) * 4))


def _profile_all(
    subset: list[SliceRecord],
    cfg,
    workers: int,
    source=None,
) -> tuple[list[SliceProfile], list[dict]]:
    """Profile every record, in parallel, preserving input order.

    Order is preserved via `executor.map` so the output is identical to the
    serial version regardless of thread scheduling -- aggregates would be the
    same either way, but a reproducible `profile.json` is worth more than the
    microseconds saved by `as_completed`.
    """
    size, vt, nr = cfg.data.img_size, cfg.data.vote_threshold, cfg.data.n_readers
    total = len(subset)

    def work(rec: SliceRecord):
        try:
            return profile_slice(rec, size, vt, nr, source), None
        except Exception as e:
            return None, {"uid": rec.uid, "error": f"{type(e).__name__}: {e}"}

    profiles: list[SliceProfile] = []
    failures: list[dict] = []
    t0 = time.time()
    step = max(total // 20, 1)

    if workers <= 1:
        results = (work(r) for r in subset)
    else:
        pool = ThreadPoolExecutor(max_workers=workers)
        results = pool.map(work, subset)

    try:
        for i, (prof, err) in enumerate(results, 1):
            if prof is not None:
                profiles.append(prof)
            else:
                failures.append(err)
            if i % step == 0 or i == total:
                dt = time.time() - t0
                rate = i / dt if dt > 0 else 0.0
                eta = (total - i) / rate if rate > 0 else 0.0
                print(
                    f"  {i}/{total}  ({100 * i / total:4.1f}%)  "
                    f"{rate:6.1f} slices/s  eta {eta / 60:4.1f} min",
                    flush=True,
                )
    finally:
        if workers > 1:
            pool.shutdown(wait=True)

    return profiles, failures


def run(
    cfg,
    max_slices: int | None = None,
    out_dir: Path | None = None,
    workers: int | None = None,
    zip_path: str | Path | None = None,
) -> dict:
    t0 = time.time()
    workers = default_workers() if workers is None else max(1, int(workers))
    out_dir = Path(out_dir or (Path(cfg.out_dir) / "profile"))
    out_dir.mkdir(parents=True, exist_ok=True)

    source = None
    if zip_path:
        from .data.zip_source import ZipSource

        print(f"[profile] reading from zip (no extraction): {zip_path}")
        source = ZipSource(zip_path)
        index_all = lambda nods: source.build_index(nods, cfg.data.n_readers)  # noqa: E731
    else:
        index_all = lambda nods: build_index(cfg.data.root, nods, cfg.data.n_readers)  # noqa: E731

    print("[profile] indexing with all nodules")
    recs_all = index_all(None)
    print("[profile] indexing with nodule-0 only")
    try:
        recs_n0 = index_all(["nodule-0"])
    except RuntimeError:
        recs_n0 = []

    counts = {
        "all_nodules": index_stats(recs_all),
        "nodule_0_only": index_stats(recs_n0) if recs_n0 else {
            "slices": 0, "patients": 0, "nodules": 0, "slices_per_patient": 0.0
        },
    }
    print(f"[profile] all nodules  : {json.dumps(counts['all_nodules'])}")
    print(f"[profile] nodule-0 only: {json.dumps(counts['nodule_0_only'])}")

    # Profile the full set unless asked to sample. Sampling is stratified by
    # patient order via a fixed stride, so it is not biased toward low ids.
    sampled = bool(max_slices and max_slices < len(recs_all))
    if sampled:
        stride = len(recs_all) / max_slices
        idx = sorted({int(i * stride) for i in range(max_slices)})
        subset = [recs_all[i] for i in idx if i < len(recs_all)]
        print(f"[profile] sampling {len(subset)} of {len(recs_all)} slices")
    else:
        subset = recs_all

    print(f"[profile] reading {len(subset)} slices x {cfg.data.n_readers + 1} files "
          f"with {workers} threads")

    # Each slice costs 5 file reads (1 image + 4 reader masks), so the full set
    # is ~29k reads of ~6KB files. On Kaggle's network-backed /kaggle/input the
    # per-file latency dominates completely -- the dataset is only 170MB, but
    # serial reads take tens of minutes. The work is I/O-bound and cv2.imread
    # releases the GIL, so threads give a near-linear speedup.
    profiles, failures = _profile_all(subset, cfg, workers, source)

    if not profiles:
        raise RuntimeError(f"profiled nothing; {len(failures)} failures, first: {failures[:1]}")

    agg = aggregate(profiles, cfg, cfg.data.n_readers)

    result = {
        "root": str(zip_path) if zip_path else str(cfg.data.root),
        "source": "zip" if zip_path else "filesystem",
        "generated": time.strftime("%Y-%m-%d %H:%M:%S"),
        "config": cfg.name,
        "slices_profiled": len(profiles),
        "sampled": sampled,
        "workers": workers,
        "failures": failures[:50],
        "n_failures": len(failures),
        "counts": counts,
        "measurements": agg,
        "minutes": round((time.time() - t0) / 60, 2),
    }

    (out_dir / "profile.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    md = write_report(result, out_dir)
    write_figures(profiles, agg, out_dir)

    if source is not None:
        source.close()

    if failures:
        print(f"[profile] {len(failures)} slices failed to read (see profile.json)")
    print(f"\n[profile] {result['minutes']} min -> {out_dir}")
    print(f"[profile] report: {md}")
    return result




# ---------------------------------------------------------------- config shim
def make_cfg(root, img_size=128, vote_threshold=2, n_readers=4,
             out_dir="/kaggle/working", name="standalone"):
    """Mirrors the lidcseg.config fields the profiler reads."""
    return SimpleNamespace(
        name=name,
        out_dir=out_dir,
        data=SimpleNamespace(
            root=str(root),
            img_size=img_size,
            vote_threshold=vote_threshold,
            n_readers=n_readers,
            nodules=None,
        ),
    )

In [ ]:
# ── 3. Resolve the slice root and profile ────────────────────────────────
# find_dataset_root digs through wrapper directories, so it copes with both
# a plain Kaggle mount and a kagglehub one (/kaggle/input/datasets/<owner>/...).
DATA_ROOT = None
for c in sorted(INPUT.glob("*")):
    try:
        DATA_ROOT = str(find_dataset_root(c))
        break
    except FileNotFoundError:
        continue

if DATA_ROOT is None:
    raise SystemExit("LIDC slices not found - add zhangweiled/lidcidri via Add Data")
print("dataset root:", DATA_ROOT)

In [ ]:
# ── 4. Run it ────────────────────────────────────────────────────────────
# WORKERS: the dataset is only ~170 MB but it is ~29,000 files of ~6 KB each,
# and /kaggle/input is network-backed, so per-file latency dominates. Reading
# serially takes tens of minutes; threads cut that to a few. Raise it if the
# rate still looks slow -- the work is I/O-bound, not CPU-bound.
WORKERS = 32

# MAX_SLICES: None profiles everything. Set 1500 for a fast strided sample
# first if you just want to see the shape of the numbers.
MAX_SLICES = None

cfg = make_cfg(DATA_ROOT, img_size=128, vote_threshold=2, n_readers=4,
               out_dir="/kaggle/working")
result = run(cfg, MAX_SLICES, pathlib.Path("/kaggle/working/profile"),
             workers=WORKERS)

In [ ]:
# ── 5. Read the report ───────────────────────────────────────────────────
from IPython.display import Image, Markdown, display

prof = pathlib.Path("/kaggle/working/profile")
display(Markdown((prof / "profile.md").read_text()))
display(Image(str(prof / "profile.png")))

In [ ]:
# ── 6. The comparison that matters ───────────────────────────────────────
ag = result["measurements"]["inter_reader_agreement"]
OLD_REPORTED = 0.8826   # best notebook cell, slice-level (leaky) split

if ag.get("n"):
    print(f"inter-reader Dice ceiling : {ag['mean']:.4f}  (sd {ag['std']:.4f})")
    print(f"old reported test Dice    : {OLD_REPORTED:.4f}")
    print()
    if OLD_REPORTED >= ag["mean"]:
        print("The old score is AT OR ABOVE the agreement between the")
        print("radiologists who drew the labels. A model cannot genuinely")
        print("segment better than its annotators agree -- that score was")
        print("fitting the vote fusion rule and the leaky split, not anatomy.")
    else:
        print(f"The old score sits {ag['mean'] - OLD_REPORTED:.4f} below the")
        print("ceiling, so there is real headroom -- but it was still measured")
        print("on a leaky split, so re-measure under a patient-grouped one.")
else:
    print("not computable - fewer than 2 reader masks found")

## Next

Copy the tables above into `docs/RESEARCH_NOTES.md` §1, replacing the
"Unverified" list. Then download `/kaggle/working/profile/` from the notebook
output panel so the numbers are kept.

Training is the separate notebook, and it **does** need Internet — Keras
downloads the ImageNet weights for the ResNet50 encoder at runtime. That needs
phone verification at kaggle.com/settings.